# glassbox — training on Colab

The runtime has its own filesystem, so this notebook clones the repository from
GitHub rather than reading your laptop. **Code has to be pushed before it can be
run here.**

Run the cells top to bottom. The first two tell you what hardware you got and
which precision path that unlocks, which is worth knowing before committing to a
long run.

### First run: restore Phase 1

Before trusting this with an overnight job, reproduce a known answer. The Phase 1
model scored **val 1.4990**, and a run that lands near it proves cloning,
installing, precision selection, Drive mounting and checkpoint return all work —
in sixteen minutes rather than six hours.

## 1 · What hardware did we get?

Colab hands out whatever is free. A T4 is Turing and has no hardware bfloat16;
an L4 or A100 does. That single fact decides which branch of `select_precision`
runs and how large a model is worth training.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no GPU visible")

import torch
print(f"torch        {torch.__version__}")
print(f"cuda         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device       {torch.cuda.get_device_name(0)}")
    print(f"bf16         {torch.cuda.is_bf16_supported()}")
    if not torch.cuda.is_bf16_supported():
        print("             -> fp16 with a gradient scaler")
    else:
        print("             -> bf16, no scaler needed")
else:
    print("\nNo GPU. Runtime > Change runtime type > GPU, then rerun.")

## 2 · Clone and install

`--no-deps` is deliberate. Colab ships a torch build matched to its CUDA driver,
and letting pip resolve dependencies risks replacing it with a generic wheel —
several minutes of download to end up with something slower or broken. Both
requirements are already present, and the cell checks rather than assumes.

In [ ]:
import os, subprocess

REPO = "https://github.com/udit-rawat/glassbox.git"
ROOT = "/content/glassbox"

if os.path.isdir(ROOT):
    print(subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT],
                         capture_output=True, text=True).stderr)

os.chdir(ROOT)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

import numpy, torch
print(f"numpy {numpy.__version__}   torch {torch.__version__}")
print(subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

## 3 · What precision will actually be used

Reading it from the same function the training loop calls, rather than guessing
from the GPU name. On the laptop this always answered `fp32`, so every other
branch has been dead code until now.

In [ ]:
from glassbox.device import get_device
from glassbox.training.precision import select_precision

device = get_device()
precision = select_precision(device, enabled=True)
print(f"device      {device}")
print(f"precision   {precision.describe()}")
print(f"scaler      {precision.use_scaler}")

## 4 · Mount Drive

Checkpoints written to the runtime's own disk vanish when the session is
recycled, which for a long run means losing the whole thing. Drive survives, and
it is what makes `--resume` meaningful: after a disconnect, rerun the training
cell with `RESUME = True` and it continues from `last.pt` rather than restarting.

Set `USE_DRIVE = False` for a short throwaway run.

In [ ]:
USE_DRIVE = True
RUN_NAME = "phase1"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/glassbox/{RUN_NAME}"
else:
    OUT_DIR = f"/content/glassbox/checkpoints/{RUN_NAME}"

os.makedirs(OUT_DIR, exist_ok=True)
print(f"checkpoints -> {OUT_DIR}")

## 5 · Sanity check

The whole suite runs on CPU in seconds. A broken environment shows up here
instead of forty minutes into a training run.

In [ ]:
print(subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--no-header"],
                     capture_output=True, text=True).stdout[-1200:])

## 6 · Train

Settings are here rather than buried in the command, so a rerun is an edit to one
cell.

**`AMP = False` for the Phase 1 restore.** The original ran in float32, and mixed
precision changes the arithmetic enough to move the final loss. Expect to land
*near* 1.4990 rather than exactly on it — different hardware reorders
floating-point operations regardless. Turn AMP on for anything new, where speed
matters more than reproducing an old number.

In [ ]:
MAX_ITERS   = 5000
LR          = 1e-3
BATCH_SIZE  = 32
GRAD_ACCUM  = 1
EVAL_EVERY  = 500
SCHEDULE    = "constant"   # constant | cosine
AMP         = False        # False reproduces the Phase 1 float32 run
RESUME      = False        # True to continue from last.pt after a disconnect

# Architecture. These defaults are the Phase 1 model; for Phase 2 use
# --norm rmsnorm --activation swiglu --pos-encoding rope --n-kv-heads 2 --no-bias
ARCH = []

cmd = [
    "python", "-u", "scripts/train_shakespeare.py",
    "--max-iters", str(MAX_ITERS),
    "--lr", str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--eval-interval", str(EVAL_EVERY),
    "--schedule", SCHEDULE,
    "--out-dir", OUT_DIR,
    "--sample-tokens", "400",
] + ARCH
if not AMP:
    cmd.append("--no-amp")
if RESUME:
    cmd.append("--resume")

print(" ".join(cmd), "\n")

# Streamed line by line so the loss curve appears as it happens. Capturing the
# output instead would show nothing until the run ends, which for a long job
# means no way to tell a slow run from a stuck one.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

## 7 · Sample from the result

Reads the tokenizer out of the checkpoint rather than assuming characters, so
this cell works unchanged once Phase 3 switches to BPE.

In [ ]:
print(subprocess.run([
    "python", "scripts/sample.py",
    "--checkpoint", f"{OUT_DIR}/best.pt",
    "--prompt", "ROMEO:",
    "--tokens", "400",
    "--temperature", "0.8",
    "--top-k", "40",
], capture_output=True, text=True).stdout)

## 8 · Confirm what came back

The checkpoint carries everything needed to rebuild the model — config, weights,
tokenizer, optimizer state — so this is also the check that it is worth
downloading before the runtime goes away.

In [ ]:
import torch
from pathlib import Path

for f in sorted(Path(OUT_DIR).iterdir()):
    print(f"{f.name:<20} {f.stat().st_size / 1e6:>8.1f} MB")

ckpt = torch.load(f"{OUT_DIR}/best.pt", map_location="cpu", weights_only=False)
cfg = ckpt["model_config"]
print(f"\niter        {ckpt['iter']}")
print(f"val loss    {ckpt['val_loss']:.4f}   (Phase 1 reference: 1.4990)")
print(f"arch        {cfg.norm} / {cfg.activation} / {cfg.pos_encoding} / kv={cfg.n_kv_heads}")
print(f"tokenizer   {ckpt['tokenizer']['kind']}")
print(f"resumable   {'optimizer' in ckpt}")

## Getting it back to the laptop

With Drive mounted the file is already synced — find it under
`MyDrive/glassbox/<run>/best.pt`. Otherwise download it directly:

```python
from google.colab import files
files.download(f"{OUT_DIR}/best.pt")
```

**Save this notebook with its outputs** before closing. The loss curve in the
cell output is the record of the run, and it is what gets read back in the repo.